# Campeón vs. subcampeón · Fórmula 1 (2000–2025)

**Visualización de Mariana — código reproducible para Google Colab**

Este cuaderno transforma la base original a una comparación de **puntos del primer y segundo lugar de cada mundial**. Usa `pandas` para procesar los datos y **Altair** para construir el gráfico interactivo. Exporta `vis_01.html` y `vis_01.jpg`.

**Origen:** `Base de datos Mariana (1).xlsx`, convertido sin añadir fuentes externas al archivo `base_mariana_para_visualizacion.csv` (se incluyen únicamente las columnas utilizadas). En la base, **no aparece el nombre del piloto en segundo lugar**, por eso la visualización muestra solamente la escudería y el puntaje de ambos puestos.

**Interacción:** al pasar el cursor por cualquiera de las dos barras de una temporada, las **dos barras** quedan semitransparentes y aparece un tooltip con la escudería y el puntaje de cada posición. Al retirar el cursor, recuperan su opacidad. La imagen JPG es necesariamente estática.

## 1. Instalar bibliotecas
La instalación es necesaria en algunas sesiones de Colab y permite exportar imágenes con `vl-convert-python`.

In [8]:
%pip -q install "altair>=5,<7" "vl-convert-python>=1.6" pandas pillow

## 2. Cargar el archivo CSV
Sube **`base_mariana_para_visualizacion.csv`** cuando Colab lo solicite. Este archivo se obtiene del Excel de Mariana y conserva los valores originales de las columnas utilizadas.

In [9]:
from google.colab import files
import io
import pandas as pd

archivos = files.upload()
nombre_csv = next((nombre for nombre in archivos if nombre.lower().endswith('.csv')), None)
if nombre_csv is None:
    raise ValueError('Sube el archivo base_mariana_para_visualizacion.csv')

original = pd.read_csv(io.BytesIO(archivos[nombre_csv]), encoding='utf-8-sig')
print('Filas originales:', len(original))
display(original.head())

Saving Base-de-datos-Mariana-_1_.csv to Base-de-datos-Mariana-_1_ (1).csv
Filas originales: 999


,Año,Piloto ganador,Equipo ganador,Puntaje,Posición 2 y puntaje,Posición 3 y puntaje,Posición 4 y puntaje,Posición 5 y puntaje,Posición 6 y puntaje,Posición 7 y puntaje,...,Posición 14 y puntaje,Posición 15 y puntaje,Posición 16 y puntaje,Posición 17 y puntaje,Posición 18 y puntaje,Posición 19 y puntaje,Posición 20 y puntaje,Posición 21 y puntaje,Posición 22 y puntaje,Posición 23 y puntaje
0,2000.0,Michael Schumacher,Ferrari,108.0,Mercedes- 89,Mercedes- 73,Ferrari- 62,BMW- 24,Benetton- 18,Honda- 17,...,Honda- 3,Benetton- 2,Supertec- 2,Petronas- 0,Cosworth- 0,Minard- 0,Minardi- 0,Peugeot- 0,Peugeot- 0,Peugeot - 0
1,2001.0,Michael Schumacher,Ferrari,123.0,Mercedes- 65,Ferrari- 56,BMW- 49,Mercedes- 37,BMW- 31,Honda- 12,...,Cosworth- 6,Acer- 6,Honda- 5,Honda- 5,Cosworth- 3,Renault- 2,Arrows- 1,NaN,NaN,NaN
2,2002.0,Michael Schumacher,Ferrari,144.0,Ferrari- 77,BMW- 50,BMW- 42,Mercedes- 41,Mercedes- 24,Renault- 14,...,Honda- 3,Honda- 2,Asiatech- 2,Toyota- 2,Asiatech- 0,Cosworth- 0,Cosworth- 0,NaN,NaN,NaN
3,2003.0,Michael Schumacher,Ferrari,93.0,Mercedes- 91,MBW- 82,Ferrari- 65,BMW- 58,Renault- 55,Mercedes- 51,...,Petronas- 6,Toyota- 6,Honda- 6,BMW- 4,Honda- 3,Ford- 1,Cosworth- 1,Cosworth- 0,Cosworth- 0,Cosworth- 0
4,2004.0,Michael Schumacher,Ferrari,148.0,Ferrari- 114,Honda- 85,Renault- 59,BMW- 58,Toyota- 46,Mercedes- 45,...,Toyota- 6,BMW- 6,Cosworth- 3,Toyota- 3,Ford- 3,Ford- 2,Cosworth- 1,Renault- 0,Toyota- 0,BMW- 0


## 3. Limpieza y filtrado

1. Normalizar nombres de columnas.
2. Limitar el período a **2000–2025** (no se incluye 2026).
3. Interpretar el número de puntos del segundo puesto de textos tales como `Mercedes- 89` o `RedBull Racing 421`. Se recupera **solo el número final**.
4. Comprobar que cada año tenga campeón y subcampeón, sin puntajes faltantes ni negativos.
5. Conservar los nombres de escudería **tal como aparecen en la base**.

In [10]:
import re
import numpy as np

base = original.copy()
base.columns = base.columns.str.strip()
columnas = ['Año', 'Equipo ganador', 'Puntaje', 'Posición 2 y puntaje']
faltantes = [c for c in columnas if c not in base.columns]
if faltantes:
    raise ValueError(f'Faltan columnas necesarias: {faltantes}')

base['Año'] = pd.to_numeric(base['Año'], errors='coerce')
base = base.loc[base['Año'].between(2000, 2025)].copy()
base['Año'] = base['Año'].astype(int)
base['Puntos primero'] = pd.to_numeric(base['Puntaje'], errors='coerce')

def extraer_segundo(texto):
    if pd.isna(texto):
        return pd.Series({'Escudería segundo': None, 'Puntos segundo': np.nan})
    texto = str(texto).strip()
    patron = r'^(.*?)[\s-]+(\d+(?:[.,]\d+)?)\s*$'
    match = re.match(patron, texto)
    if match is None:
        raise ValueError(f'No se pudo leer segundo lugar: {texto!r}')
    nombre = match.group(1).rstrip(' -').strip()
    puntos = float(match.group(2).replace(',', '.'))
    return pd.Series({'Escudería segundo': nombre, 'Puntos segundo': puntos})

base[['Escudería segundo', 'Puntos segundo']] = base['Posición 2 y puntaje'].apply(extraer_segundo)
base['Escudería primero'] = base['Equipo ganador'].astype(str).str.strip()
base['Brecha (puntos)'] = base['Puntos primero'] - base['Puntos segundo']

assert len(base) == 26, 'Se esperaban 26 temporadas, de 2000 a 2025.'
assert base['Año'].is_unique, 'Existen años duplicados.'
assert not base[['Puntos primero', 'Puntos segundo', 'Escudería primero', 'Escudería segundo']].isna().any().any()
assert (base['Brecha (puntos)'] >= 0).all(), 'El segundo puesto no puede superar al primero.'
base = base.sort_values('Año').reset_index(drop=True)
display(base[['Año','Escudería primero','Puntos primero','Escudería segundo','Puntos segundo','Brecha (puntos)']].head())

,Año,Escudería primero,Puntos primero,Escudería segundo,Puntos segundo,Brecha (puntos)
0,2000,Ferrari,108.0,Mercedes,89.0,19.0
1,2001,Ferrari,123.0,Mercedes,65.0,58.0
2,2002,Ferrari,144.0,Ferrari,77.0,67.0
3,2003,Ferrari,93.0,Mercedes,91.0,2.0
4,2004,Ferrari,148.0,Ferrari,114.0,34.0


## 4. Preparar barras y tooltip
Usaremos dos registros por temporada. Para lograr un **solo recuadro con información de ambas posiciones**, duplicamos en cada barra los campos de primer y segundo lugar. Así, al apuntar a cualquiera de las dos, el tooltip muestra el par completo.

In [11]:
primero = base.assign(Posición='1.er lugar', Puntos=base['Puntos primero'])
segundo = base.assign(Posición='2.º lugar', Puntos=base['Puntos segundo'])

larga = pd.concat([primero, segundo], ignore_index=True)
larga['Temporada'] = larga['Año'].astype(str)
larga['Posición'] = pd.Categorical(
    larga['Posición'], categories=['1.er lugar', '2.º lugar'], ordered=True
)

display(larga[['Temporada','Posición','Puntos','Escudería primero','Puntos primero','Escudería segundo','Puntos segundo']].head(6))

,Temporada,Posición,Puntos,Escudería primero,Puntos primero,Escudería segundo,Puntos segundo
0,2000,1.er lugar,108.0,Ferrari,108.0,Mercedes,89.0
1,2001,1.er lugar,123.0,Ferrari,123.0,Mercedes,65.0
2,2002,1.er lugar,144.0,Ferrari,144.0,Ferrari,77.0
3,2003,1.er lugar,93.0,Ferrari,93.0,Mercedes,91.0
4,2004,1.er lugar,148.0,Ferrari,148.0,Ferrari,114.0
5,2005,1.er lugar,133.0,Renault,133.0,Mercedes,112.0


## 5. Gráfico interactivo en Altair

- Dos barras por temporada: **rojo, primero; azul, segundo**.
- Todas mantienen su color y opacidad original cuando no hay hover.
- `selection_point(fields=['Año'])`: la selección se realiza **por año**, de modo que el hover en cualquiera de las barras afecta a **ambas barras**.
- `condition`: al seleccionar un año, el par de barras queda al **50 % de opacidad**.
- `tooltip`: un mismo recuadro flotante con escudería y puntaje del primer y segundo lugar.

*Nota:* el rango de puntos es compartido en el período, pero los sistemas de puntuación históricos cambiaron; no se debe interpretar la altura absoluta como medida de dominio entre temporadas.

In [12]:
import altair as alt

alt.data_transformers.disable_max_rows()
hover_anio = alt.selection_point(
    name='hover_anio',
    fields=['Año'],
    on='pointerover',
    clear='pointerout',
    empty=False
)

orden = ['1.er lugar', '2.º lugar']

chart = (
    alt.Chart(larga)
    .mark_bar(cornerRadiusTopLeft=2, cornerRadiusTopRight=2)
    .encode(
        x=alt.X('Temporada:N', sort=[str(a) for a in range(2000, 2026)], title='Temporada', axis=alt.Axis(labelAngle=-55)),
        xOffset=alt.XOffset('Posición:N', sort=orden),
        y=alt.Y('Puntos:Q', title='Puntos del campeonato', scale=alt.Scale(zero=True)),
        color=alt.Color('Posición:N', scale=alt.Scale(domain=orden, range=['#E45D6A','#469ACB']), legend=alt.Legend(title=None, orient='top')),
        opacity=alt.condition(hover_anio, alt.value(0.5), alt.value(1)),
        tooltip=[
            alt.Tooltip('Año:Q', title='Temporada', format='.0f'),
            alt.Tooltip('Escudería primero:N', title='1.er lugar · escudería'),
            alt.Tooltip('Puntos primero:Q', title='1.er lugar · puntos', format=',.1f'),
            alt.Tooltip('Escudería segundo:N', title='2.º lugar · escudería'),
            alt.Tooltip('Puntos segundo:Q', title='2.º lugar · puntos', format=',.1f'),
            alt.Tooltip('Brecha (puntos):Q', title='Diferencia de puntos', format=',.1f')
        ]
    )
    .add_params(hover_anio)
    .properties(
        title=alt.TitleParams('Campeón vs. subcampeón', subtitle=['Fórmula 1 · 2000–2025 | Puntaje de los dos primeros lugares']),
        width=1050,
        height=440
    )
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=11, titleFontSize=13, gridColor='#E6EAF0')
    .configure_title(fontSize=23, subtitleFontSize=12, anchor='start')
)

chart

alt.Chart(...)

## 6. Exportación a HTML y JPG

- **HTML:** mantiene tooltips y efectos al pasar el cursor.
- **JPG:** copia estática para la entrega y la webstory. Altair exporta el gráfico primero a PNG (sin pérdida) y luego Pillow lo convierte a JPG porque JPEG no soporta transparencia ni interactividad.
- Se crea además `datos_limpios.csv` para que se pueda auditar el procesamiento.

In [6]:
from PIL import Image
from pathlib import Path

carpeta = Path('visualizacion')
carpeta.mkdir(exist_ok=True)

ruta_html = carpeta / 'vis_01.html'
ruta_png = carpeta / 'vis_01_temporal.png'
ruta_jpg = carpeta / 'vis_01.jpg'

chart.save(str(ruta_html))
chart.save(str(ruta_png), scale_factor=2)
with Image.open(ruta_png) as img:
    img.convert('RGB').save(ruta_jpg, 'JPEG', quality=94, optimize=True)
ruta_png.unlink()

base[['Año','Escudería primero','Puntos primero','Escudería segundo','Puntos segundo','Brecha (puntos)']].to_csv('datos_limpios.csv', index=False, encoding='utf-8-sig')
print(f'HTML: {ruta_html}')
print(f'JPG:  {ruta_jpg}')
print('CSV limpio: datos_limpios.csv')

HTML: visualizacion/vis_01.html
JPG:  visualizacion/vis_01.jpg
CSV limpio: datos_limpios.csv


## 7. Descargar los archivos
Puedes descargar juntos los resultados y ubicarlos dentro de `visualizacion/` en el repositorio. El cuaderno `.ipynb` se guarda en `scripts/`.

In [7]:
from google.colab import files
files.download('visualizacion/vis_01.html')
files.download('visualizacion/vis_01.jpg')
files.download('datos_limpios.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Estructura sugerida para la entrega

```text
Apellido_Mariana_Campeon_vs_subcampeon/
├── README.md
├── Ficha_Técnica.md
├── base_mariana_para_visualizacion.csv
├── visualizacion/
│   ├── vis_01.html
│   ├── vis_01.jpg
│   └── cronica.md
└── scripts/
    └── codigo_para_visualizar_1.ipynb
```

**Metodología:** el gráfico usa exclusivamente los campos `Año`, `Equipo ganador`, `Puntaje` y `Posición 2 y puntaje` de la base de Mariana. Se preservan las etiquetas de escudería originales, sin unir nombres equivalentes, porque no fue proporcionada una tabla de homologación. Un gráfico de puntajes absolutos permite comparar a los dos pilotos **dentro del mismo año**; no normaliza cambios históricos de puntuación.